# Chapter 18. Speed, Accuracy, and Validation

The code from Chapter 18 of *Radio Propagation Ray Tracing with Sionna RT*.
The cells follow the order of the text, and the contents of each code cell are
the same as printed there.

Only the first Setup cell is not printed in the book. It performs the imports
and the scene loading that the printed code assumes. Once the setup cell has
run, the rest can be run from the top in order.

Fragments the text shows for explanation, such as a path to a scene of your own
or an object name that scene does not contain, will not run as they stand. A
note appears before each such cell.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (load_scene, scene as scene_assets,
                       Transmitter, Receiver, PlanarArray,
                       PathSolver, RadioMapSolver, Camera, RadioMaterial)

# the central Munich urban scene used in many chapters of this book
scene = load_scene(scene_assets.munich)
scene.frequency = 3.5e9
scene.tx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.rx_array = PlanarArray(num_rows=1, num_cols=1,
                             pattern='iso', polarization='V')
scene.add(Transmitter('tx0', position=[-30.0, 38.0, 35.0]))

solver = PathSolver()
rmsolver = RadioMapSolver()

## 2. The Effect of the Maximum Interaction Count

In [ ]:
import numpy as np
from sionna.rt import PathSolver

solver = PathSolver()
for depth in [0, 1, 3]:
    paths = solver(scene, max_depth=depth, los=True,
                   specular_reflection=True, seed=42)
    a = np.array(paths.a[0]) + 1j*np.array(paths.a[1])
    print(depth, int(np.array(paths.valid).sum()),
          10*np.log10(np.sum(np.abs(a)**2)))

## 3. The Effect of the Samples per Transmitter

In [ ]:
import numpy as np
from sionna.rt import RadioMapSolver

rms = RadioMapSolver()
for n in [10**4, 10**5, 10**6]:
    rm = rms(scene, max_depth=5, cell_size=[5.0, 5.0],
             samples_per_tx=n, seed=42)
    pg = np.array(rm.path_gain)[0]
    print(n, 'unreached fraction %.3f' % np.mean(pg == 0))

## 4. The Effect of Cell Size

In [ ]:
for cell in [1.0, 3.0, 10.0]:
    rm = rms(scene, max_depth=3, cell_size=[cell, cell],
             samples_per_tx=10**5, seed=42)
    print(cell, np.array(rm.path_gain).shape)   # (Tx, y, x)

```bash
# record the usage from another terminal while the computation runs
nvidia-smi --query-gpu=memory.used --format=csv,noheader -l 1 \
  | tee gpu_mem.log
```

## 7. Comparison Against Free-Space Path Loss

In [ ]:
from sionna.rt import (load_scene, PlanarArray, Transmitter,
                       Receiver, PathSolver)

scene = load_scene()          # no argument = an empty scene with no objects
scene.frequency = 3.5e9
ant = PlanarArray(num_rows=1, num_cols=1,
                  pattern='iso', polarization='V')
scene.tx_array = ant
scene.rx_array = ant
scene.add(Transmitter('tx', position=[0.0, 0.0, 30.0]))
scene.add(Receiver('rx', position=[100.0, 0.0, 30.0]))

In [ ]:
import numpy as np

paths = PathSolver()(scene, max_depth=0, los=True,
                     specular_reflection=False)
a = np.array(paths.a[0]) + 1j*np.array(paths.a[1])
print('Sionna RT:', 10*np.log10(np.sum(np.abs(a)**2)))
print('theory   :', -(20*np.log10(100.0)
                      + 20*np.log10(3.5e9) - 147.55))

## 8. Comparison Against the Two-Ray Model

In [ ]:
from sionna.rt import load_scene_from_string

GROUND_XML = """<scene version="2.1.0">
  <bsdf type="itu-radio-material" id="mat">
    <string name="type" value="concrete"/>
  </bsdf>
  <shape type="rectangle" id="ground">
    <transform name="to_world"><scale value="3000"/></transform>
    <ref id="mat" name="bsdf"/>
  </shape>
</scene>"""

scene = load_scene_from_string(GROUND_XML)   # 6 km square of ground alone
scene.frequency = 3.5e9

In [ ]:
paths = PathSolver()(scene, max_depth=1, los=True,
                     specular_reflection=True, seed=42)
a = np.array(paths.a[0]) + 1j*np.array(paths.a[1])
print('paths:', int(np.array(paths.valid).sum()))   # comes out as 2
print('gain [dB]:', 10*np.log10(np.sum(np.abs(a)**2)))